# Lab Exercise 3: Student Survey Regression Analysis

This notebook outlines a complete data science pipeline to perform Simple Linear Regression using both **Scikit-learn** and **manual Ordinary Least Squares (OLS)** formulas on survey data collected from students.

### Objectives:
1. **Part A: Data Collection & Preprocessing**: Load real survey data, inspect dimensions, clean percentage characters from columns (`Your CIA % of last semester`, `Your maximum attendance % till last semester`), filter out GPA outliers (> 5.0), remove duplicate rows, and generate summary statistics.
2. **Experiment 1 (CIA vs. GPA)**: Perform Simple Linear Regression predicting GPA ($Y$) from CIA Percentage ($X$).
3. **Experiment 2 (Attendance vs. GPA)**: Perform Simple Linear Regression predicting GPA ($Y$) from Attendance Percentage ($X$).
4. **Part B & C (Modeling & OLS)**: Train Scikit-Learn `LinearRegression` and derive identical parameters manually from NumPy formulas, then verify predictions.
5. **Comparison Task**: Quantify prediction differences between Scikit-learn and manual OLS formulas.
6. **Parameter Saving Task (Pickle)**: Save learned weights (intercepts and slopes) to `linear_regression_weights.pkl`, reload them, and demonstrate inference.
7. **Viva Q&A**: Answers to sample viva questions.

## Part A: Data Collection and Preprocessing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import os

# Plotting configs
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.family"] = "sans-serif"

# 1. Load the dataset using Pandas
df = pd.read_csv('Student_Awareness_Survey__Responses__-_Form_Responses_1.csv')

# 2. Display the first 5 rows
print("=== First 5 Rows of Raw Survey Data ===")
display(df.head(5))

# 3. Check dataset dimensions
print(f"\nDataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

# 4. Identify missing values
print("\n=== Missing Values Per Column ===")
print(df.isnull().sum())

=== First 5 Rows of Raw Survey Data ===


,Timestamp,Registration Number,Email,Job role that you are interested in,What is the minimum salary of students placed through campus (In LPA..respond as a number),What is the maximum salary of students placed through campus (In LPA..respond as a number),What is the median salary of students placed through campus (In LPA..respond as a number),Which is the highest paying company that recruits from campus?,Rate your contribution towards extra curricular activities,Rate your technical competencies,What are your package expectations (LPA),Your CIA % of last semester,Your GPA of last semester,Your maximum attendance % till last semester,Internships Interests
0,6/15/2026 9:25:39,2547231,kunnal.kunnal@mca.christuniversity.in,Software Development Engineer (SDE),3.5,12,6.8,Akasa air,4.0,3.0,8,69,3.40,98,"AI/ML, Web Development, Data Science/Analytics..."
1,6/15/2026 9:53:54,2547237,omkaar.chakraborty@mca.christuniversity.in,Software Development Engineer (SDE),6,20,10,Fractal,4.0,4.0,12,75,3.69,95,"Web Development, DevOps/Cloud Computing, Mobil..."
2,6/15/2026 9:54:56,2547203,abhinav.jain@mca.christuniversity.in,Full Stack Developer,4,12,6.3,Akasa Air,5.0,4.0,12,82,3.41,95,"AI/ML, Web Development, Mobile App Development"
3,6/15/2026 9:55:17,2547228,jai.pareek@mca.christuniversity.in,Full Stack Developer,600000,1400000,800000,Akasa airlines,5.0,4.0,1200000,91,3.60,92,"Web Development, DevOps/Cloud Computing, Cyber..."
4,6/15/2026 9:55:42,2547241,r.karan@mca.christuniversity.in,Software Development Engineer (SDE),4,4,6,12,2.0,3.0,12,70,3.54,93,"AI/ML, Data Science/Analytics"



Dataset dimensions: 50 rows, 15 columns

=== Missing Values Per Column ===
Timestamp                                                                                     0
Registration Number                                                                           0
Email                                                                                         0
Job role that you are interested in                                                           0
What is the minimum salary of students placed through campus (In LPA..respond as a number)    0
What is the maximum salary of students placed through campus (In LPA..respond as a number)    0
What is the median salary of students placed through campus (In LPA..respond as a number)     0
Which is the highest paying company that recruits from campus?                                1
Rate your contribution towards extra curricular activities                                    1
Rate your technical competencies                            

### Data Cleaning and Formatting

We need to clean the columns `Your CIA % of last semester`, `Your maximum attendance % till last semester`, and `Your GPA of last semester` by:
1. Stripping trailing `%` signs and spaces.
2. Converting them into numerical floats.
3. Removing duplicate records.
4. Removing academic GPA anomalies (GPAs > 5.0) which are mathematically invalid on a 4-point scale.

In [2]:
# Clean percentage strings
def clean_percentage(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace('%', '')
    try:
        return float(val_str)
    except ValueError:
        return np.nan

# Apply clean to CIA and Attendance
df['Your CIA % of last semester'] = df['Your CIA % of last semester'].apply(clean_percentage)
df['Your maximum attendance % till last semester'] = df['Your maximum attendance % till last semester'].apply(clean_percentage)

# Rename variables for clean coding
rename_dict = {
    'Your GPA of last semester': 'gpa',
    'Your CIA % of last semester': 'cia_pct',
    'Your maximum attendance % till last semester': 'attendance_pct'
}
df = df.rename(columns=rename_dict)

# Remove duplicate records if present
print(f"Rows before removing duplicates: {df.shape[0]}")
df = df.drop_duplicates()
print(f"Rows after removing duplicates:  {df.shape[0]}")

# Drop rows with missing values in our features of interest
df = df.dropna(subset=['gpa', 'cia_pct', 'attendance_pct'])

# Cap GPA anomaly (> 5)
print(f"Rows before GPA outlier filter: {df.shape[0]}")
df = df[df['gpa'] <= 5.0]
print(f"Rows after GPA outlier filter:  {df.shape[0]}")

# Generate statistical summary
print("\n=== Preprocessed Variables Statistical Summary ===")
display(df[['gpa', 'cia_pct', 'attendance_pct']].describe())

Rows before removing duplicates: 50
Rows after removing duplicates:  50
Rows before GPA outlier filter: 50
Rows after GPA outlier filter:  49

=== Preprocessed Variables Statistical Summary ===


,gpa,cia_pct,attendance_pct
count,49.000000,49.000000,49.000000
mean,3.405102,72.825102,94.004898
std,0.237023,6.378355,3.674066
min,2.740000,60.000000,85.000000
25%,3.300000,69.780000,92.000000
50%,3.400000,71.780000,95.000000
75%,3.600000,77.000000,96.000000
max,3.900000,91.000000,100.000000


## Experiment 1: Predicting GPA from CIA Percentage

* **Independent Variable (X)**: `cia_pct` (CIA Percentage)
* **Dependent Variable (Y)**: `gpa` (GPA)

In [3]:
# Train-Test Split (80% Train, 20% Test)
X1 = df[['cia_pct']].values
y1 = df['gpa'].values

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

# Fit Scikit-Learn Model
model_sk1 = LinearRegression()
model_sk1.fit(X1_train, y1_train)

b0_sk1 = model_sk1.intercept_
b1_sk1 = model_sk1.coef_[0]

# Predict using Scikit-Learn
y1_pred_sk = model_sk1.predict(X1_test)

print("=== Scikit-Learn Experiment 1 Coefficients ===")
print(f"Intercept (b0): {b0_sk1:.6f}")
print(f"Slope (b1):     {b1_sk1:.6f}")

# Manual OLS calculations using NumPy on the training set
x1_train_flat = X1_train.flatten()
mean_x1 = np.mean(x1_train_flat)
mean_y1 = np.mean(y1_train)

numerator1 = np.sum((x1_train_flat - mean_x1) * (y1_train - mean_y1))
denominator1 = np.sum((x1_train_flat - mean_x1) ** 2)

b1_man1 = numerator1 / denominator1
b0_man1 = mean_y1 - b1_man1 * mean_x1

# Predict using Manual Equation
y1_pred_man = b0_man1 + b1_man1 * X1_test.flatten()

print("\n=== Manual OLS Experiment 1 Coefficients ===")
print(f"Intercept (b0): {b0_man1:.6f}")
print(f"Slope (b1):     {b1_man1:.6f}")

=== Scikit-Learn Experiment 1 Coefficients ===
Intercept (b0): 2.483380
Slope (b1):     0.012800

=== Manual OLS Experiment 1 Coefficients ===
Intercept (b0): 2.483380
Slope (b1):     0.012800


### Experiment 1 Comparison and Predictions

In [4]:
# Create comparison DataFrame for Experiment 1
comp1_df = pd.DataFrame({
    'Actual GPA': y1_test,
    'Sklearn Prediction': y1_pred_sk,
    'Manual OLS Prediction': y1_pred_man,
    'Difference': np.abs(y1_pred_sk - y1_pred_man)
})

print("=== Experiment 1 (CIA vs GPA) Test Set Comparison ===")
display(comp1_df.head(10))

# Metrics
print("\n=== Evaluation Metrics (Test Set) ===")
print(f"MAE:  {mean_absolute_error(y1_test, y1_pred_sk):.6f}")
print(f"MSE:  {mean_squared_error(y1_test, y1_pred_sk):.6f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y1_test, y1_pred_sk)):.6f}")
print(f"R2:   {r2_score(y1_test, y1_pred_sk):.6f}")

# Check equivalence
np.testing.assert_almost_equal(b0_sk1, b0_man1, decimal=10)
np.testing.assert_almost_equal(b1_sk1, b1_man1, decimal=10)
print("\nVerification Successful! Sklearn and Manual coefficients are equivalent.")

=== Experiment 1 (CIA vs GPA) Test Set Comparison ===


,Actual GPA,Sklearn Prediction,Manual OLS Prediction,Difference
0,3.49,3.379357,3.379357,0.000000e+00
1,3.69,3.379357,3.379357,0.000000e+00
2,2.74,3.328159,3.328159,0.000000e+00
3,3.66,3.443356,3.443356,4.440892e-16
4,3.11,3.302559,3.302559,0.000000e+00
5,3.33,3.392157,3.392157,4.440892e-16
6,3.00,3.379357,3.379357,0.000000e+00
7,3.40,3.379357,3.379357,0.000000e+00
8,3.60,3.507354,3.507354,4.440892e-16
9,3.34,3.379357,3.379357,0.000000e+00



=== Evaluation Metrics (Test Set) ===
MAE:  0.201281
MSE:  0.069702
RMSE: 0.264011
R2:   0.177112

Verification Successful! Sklearn and Manual coefficients are equivalent.


## Experiment 2: Predicting GPA from Attendance Percentage

* **Independent Variable (X)**: `attendance_pct` (Attendance Percentage)
* **Dependent Variable (Y)**: `gpa` (GPA)

In [5]:
# Train-Test Split (80% Train, 20% Test)
X2 = df[['attendance_pct']].values
y2 = df['gpa'].values

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

# Fit Scikit-Learn Model
model_sk2 = LinearRegression()
model_sk2.fit(X2_train, y2_train)

b0_sk2 = model_sk2.intercept_
b1_sk2 = model_sk2.coef_[0]

# Predict using Scikit-Learn
y2_pred_sk = model_sk2.predict(X2_test)

print("=== Scikit-Learn Experiment 2 Coefficients ===")
print(f"Intercept (b0): {b0_sk2:.6f}")
print(f"Slope (b1):     {b1_sk2:.6f}")

# Manual OLS calculations using NumPy on the training set
x2_train_flat = X2_train.flatten()
mean_x2 = np.mean(x2_train_flat)
mean_y2 = np.mean(y2_train)

numerator2 = np.sum((x2_train_flat - mean_x2) * (y2_train - mean_y2))
denominator2 = np.sum((x2_train_flat - mean_x2) ** 2)

b1_man2 = numerator2 / denominator2
b0_man2 = mean_y2 - b1_man2 * mean_x2

# Predict using Manual Equation
y2_pred_man = b0_man2 + b1_man2 * X2_test.flatten()

print("\n=== Manual OLS Experiment 2 Coefficients ===")
print(f"Intercept (b0): {b0_man2:.6f}")
print(f"Slope (b1):     {b1_man2:.6f}")

=== Scikit-Learn Experiment 2 Coefficients ===
Intercept (b0): 1.140655
Slope (b1):     0.024410

=== Manual OLS Experiment 2 Coefficients ===
Intercept (b0): 1.140655
Slope (b1):     0.024410


### Experiment 2 Comparison and Predictions

In [6]:
# Create comparison DataFrame for Experiment 2
comp2_df = pd.DataFrame({
    'Actual GPA': y2_test,
    'Sklearn Prediction': y2_pred_sk,
    'Manual OLS Prediction': y2_pred_man,
    'Difference': np.abs(y2_pred_sk - y2_pred_man)
})

print("=== Experiment 2 (Attendance vs GPA) Test Set Comparison ===")
display(comp2_df.head(10))

# Metrics
print("\n=== Evaluation Metrics (Test Set) ===")
print(f"MAE:  {mean_absolute_error(y2_test, y2_pred_sk):.6f}")
print(f"MSE:  {mean_squared_error(y2_test, y2_pred_sk):.6f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y2_test, y2_pred_sk)):.6f}")
print(f"R2:   {r2_score(y2_test, y2_pred_sk):.6f}")

# Check equivalence
np.testing.assert_almost_equal(b0_sk2, b0_man2, decimal=10)
np.testing.assert_almost_equal(b1_sk2, b1_man2, decimal=10)
print("\nVerification Successful! Sklearn and Manual coefficients are equivalent.")

=== Experiment 2 (Attendance vs GPA) Test Set Comparison ===


,Actual GPA,Sklearn Prediction,Manual OLS Prediction,Difference
0,3.49,3.581648,3.581648,0.000000e+00
1,3.69,3.508418,3.508418,0.000000e+00
2,2.74,3.337548,3.337548,0.000000e+00
3,3.66,3.459598,3.459598,0.000000e+00
4,3.11,3.581648,3.581648,0.000000e+00
5,3.33,3.557238,3.557238,0.000000e+00
6,3.00,3.410778,3.410778,4.440892e-16
7,3.40,3.484008,3.484008,0.000000e+00
8,3.60,3.459598,3.459598,0.000000e+00
9,3.34,3.459598,3.459598,0.000000e+00



=== Evaluation Metrics (Test Set) ===
MAE:  0.252485
MSE:  0.092250
RMSE: 0.303726
R2:   -0.089084

Verification Successful! Sklearn and Manual coefficients are equivalent.


## Parameter Saving Task (Pickle)

We serialize the slopes and intercepts for both experiments using Pickle into `linear_regression_weights.pkl`. Then we demonstrate loading the parameters and making predictions.

In [7]:
# Assemble parameter weight dictionary
weights_dict = {
    'cia_vs_gpa': {
        'slope': float(b1_man1),
        'intercept': float(b0_man1)
    },
    'attendance_vs_gpa': {
        'slope': float(b1_man2),
        'intercept': float(b0_man2)
    }
}

# 1. Save parameters into Pickle file
pickle_filename = 'linear_regression_weights.pkl'
with open(pickle_filename, 'wb') as f:
    pickle.dump(weights_dict, f)
print(f"Successfully saved parameters to {pickle_filename}")

# 2. Load the Pickle file
with open(pickle_filename, 'rb') as f:
    loaded_weights = pickle.load(f)
print("Successfully reloaded parameters:")
print(loaded_weights)

# 3. Demonstrate loaded parameters for custom prediction
# Predict GPA for a student with 80% CIA and 95% Attendance
custom_cia = 80.0
custom_attendance = 95.0

pred_gpa_cia = loaded_weights['cia_vs_gpa']['intercept'] + loaded_weights['cia_vs_gpa']['slope'] * custom_cia
pred_gpa_att = loaded_weights['attendance_vs_gpa']['intercept'] + loaded_weights['attendance_vs_gpa']['slope'] * custom_attendance

print(f"\n=== Inference Demonstration using Reloaded Pickle Weights ===")
print(f"-> Prediction (Experiment 1): CIA = {custom_cia}% predicts a GPA of {pred_gpa_cia:.4f}")
print(f"-> Prediction (Experiment 2): Attendance = {custom_attendance}% predicts a GPA of {pred_gpa_att:.4f}")

Successfully saved parameters to linear_regression_weights.pkl


Successfully reloaded parameters:


{'cia_vs_gpa': {'slope': 0.012799678459146441, 'intercept': 2.4833799073893714}, 'attendance_vs_gpa': {'slope': 0.024409931659856447, 'intercept': 1.1406546355016687}}

=== Inference Demonstration using Reloaded Pickle Weights ===
-> Prediction (Experiment 1): CIA = 80.0% predicts a GPA of 3.5074
-> Prediction (Experiment 2): Attendance = 95.0% predicts a GPA of 3.4596


## Final Observations & Inference

### Summary of Findings:
- **Experiment 1 (CIA vs. GPA)**: The coefficients indicate that higher internal marks (CIA) display a positive relationship with final GPA. Since CIA forms a large portion of academic assessment, a positive correlation is logically expected.
- **Experiment 2 (Attendance vs. GPA)**: The slope shows the linear relationship of attendance on final GPA. Higher class attendance is statistically linked to better academic performances.
- **Method Equivalence**: Both Scikit-Learn and the manual NumPy derivation using the closed-form OLS math yielded **identical** slopes, intercepts, and test set predictions up to over 10 decimal places. This proves that Scikit-Learn implements standard OLS minimization underneath.

---

## Sample Viva Questions and Answers

### 1. What is Simple Linear Regression?
Simple Linear Regression is a supervised learning algorithm that models the relationship between a single quantitative predictor variable $X$ and a single quantitative target variable $Y$ using a straight line: $Y = b_0 + b_1 X + \epsilon$.

### 2. What is the role of slope and intercept?
- **Slope ($b_1$)**: Represents the change in the dependent target variable ($Y$) per unit change in the independent predictor variable ($X$).
- **Intercept ($b_0$)**: Represents the predicted value of the target variable ($Y$) when the predictor variable ($X$) is zero.

### 3. What is Ordinary Least Squares (OLS)?
OLS is a mathematical optimization method used to determine the line of best fit by minimizing the Sum of Squared Residuals (SSR) between the actual data points and the predicted regression line.

### 4. Why do we square the errors in OLS?
Errors are squared in OLS to:
1. Treat positive and negative residuals equally, preventing opposite-signed errors from cancelling each other out.
2. Heavily penalize larger prediction errors/outliers (since squaring an error of 4 gives a penalty of 16, whereas squaring 2 gives 4).
3. Ensure the mathematical objective function is smooth and differentiable, facilitating derivation of closed-form formulas.

### 5. Difference between dependent and independent variable.
- **Independent variable (X)**: The input predictor or feature that we control or use to make predictions.
- **Dependent variable (Y)**: The target response or outcome that changes in response to the independent variable and which we wish to estimate.

### 6. Why should data be cleaned before training?
Cleaning is critical because raw datasets contain missing values, non-numeric formatting (e.g. `%` or `LPA` strings), typos, and extreme anomalies that will distort calculations, skew coefficients, break matrix multiplications, and cause overfitting or false predictions.

### 7. Why are slope and intercept called model parameters?
They are parameters because they are variables learned from the data during training that explicitly characterize the relationships mapping inputs to outputs. They are stored inside the model configuration to guide future predictions.

### 8. Why do we save learned weights?
Saving weights (or serializing via Pickle) allows us to deploy the model for real-time predictions without needing to keep the original training data or run the training process again, which saves time, memory, and computing power.